# Retail Sales and Profitability Analysis

**Data analytics portfolio · Python, pandas, Matplotlib**

This project examines transaction-level retail performance and turns the results into questions for business review. The notebook combines data inspection with exploratory analysis and uses a consistent reporting population throughout.

## Business questions

1. How did sales and profit vary across product categories?
2. Which sub-categories generated the most profit or loss?
3. How were discounts associated with profit?

**Grain:** one row represents a product line within an order. Repeated order IDs are expected.


## Method and interpretation

The workflow validates the source, parses dates, applies the documented reporting cutoff, and then compares sales, profit, and weighted profit margin across business dimensions. The reporting cutoff used in this project is **November 9, 2026**; 2026 is therefore a partial year. Change the cutoff only when the project brief supports a different period.

Discount and shipping comparisons are descriptive associations. They do not prove cause and effect. Negative profit is treated as a valid business result.


### Load the source data


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Support running from the project root or from the notebooks folder.
candidates = [Path("data/SampleSuperstore.csv"), Path("../data/SampleSuperstore.csv")]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/SampleSuperstore.csv")
df = pd.read_csv(data_path, encoding="latin1")
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {data_path.resolve()}")
display(df.head())


### Dataset profile


In [ ]:
profile = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "unique_values": df.nunique(dropna=True),
})
profile.index.name = "column"
print(f"Dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
display(profile)


### Inspect the dataset schema


## Data quality and completeness


### Expected fields and data types


In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

print("Invalid Order Dates:", df["Order Date"].isna().sum())
print("Invalid Ship Dates:", df["Ship Date"].isna().sum())

print("Orders shipped before ordering:",
      (df["Ship Date"] < df["Order Date"]).sum())


### Business rule checks
Flag potential issues for review. These checks do not remove rows, and negative profit remains a valid business outcome.

In [ ]:
rule_checks = pd.Series({
    "ship date before order date": int((df["Ship Date"] < df["Order Date"]).sum()),
    "negative sales rows": int((df["Sales"] < 0).sum()),
    "zero or negative quantity rows": int((df["Quantity"] <= 0).sum()),
    "discount outside 0–1 rows": int((~df["Discount"].between(0, 1)).sum()),
    "missing order IDs": int(df["Order ID"].isna().sum()),
    "missing product IDs": int(df["Product ID"].isna().sum()),
})
display(rule_checks.rename("flagged_rows").to_frame())


### Parse and validate order and ship dates


## Define the reporting period


In [ ]:
# Keep records up to November 9, 2026
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
analysis_df = df[
    df["Order Date"] <= pd.Timestamp("2026-11-09")
].copy()

print("Original records:", len(df))
print("Historical records:", len(analysis_df))

print("Historical unique orders:",
      analysis_df["Order ID"].nunique())

print("Historical sales:",
      round(analysis_df["Sales"].sum(), 2))

print("Historical profit:",
      round(analysis_df["Profit"].sum(), 2))

print("Historical date range:",
      analysis_df["Order Date"].min(),
      "to",
      analysis_df["Order Date"].max())



## Annual performance


In [ ]:
#Recalculate annual performance
annual_analysis = analysis_df.groupby(
    analysis_df["Order Date"].dt.year
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")

)

annual_analysis["Profit_Margin_%"] = (
    annual_analysis["Total_Profit"] /
    annual_analysis["Total_Sales"] * 100
)
annual_analysis["Sales_Growth_%"] = (
    annual_analysis["Total_Sales"].pct_change() * 100
)
annual_analysis.round(2)



### Comparable year-to-date performance


In [ ]:
ytd_comparison = analysis_df[
    (
        (analysis_df["Order Date"] >= "2025-01-01") &
        (analysis_df["Order Date"] <= "2025-11-09")
    ) |
    (
        (analysis_df["Order Date"] >= "2026-01-01") &
        (analysis_df["Order Date"] <= "2026-11-09")
    )
].copy()

ytd_analysis = ytd_comparison.groupby(
    ytd_comparison["Order Date"].dt.year
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

ytd_analysis["Profit_Margin_%"] = (
    ytd_analysis["Total_Profit"] /
    ytd_analysis["Total_Sales"] * 100
)

ytd_analysis["Sales_Growth_%"] = (
    ytd_analysis["Total_Sales"].pct_change() * 100
)

ytd_analysis.round(2)


## Category performance


In [ ]:
#Product Category Analysis

category_analysis = analysis_df.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

category_analysis["Profit_Margin_%"] = (
    category_analysis["Total_Profit"] /
    category_analysis["Total_Sales"] * 100
)

category_analysis.sort_values(
    "Total_Profit", ascending=False
).round(2)



### Sub-category performance


In [ ]:
subcategory_analysis = analysis_df.groupby("Sub-Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)

subcategory_analysis["Profit_Margin_%"] = (
    subcategory_analysis["Total_Profit"] /
    subcategory_analysis["Total_Sales"] * 100
)

subcategory_analysis = subcategory_analysis.sort_values(
    "Profit_Margin_%", ascending=False
).round(2)

display(subcategory_analysis)

print("Total Sales:", subcategory_analysis["Total_Sales"].sum())
print("Total Profit:", subcategory_analysis["Total_Profit"].sum())


## Product-level loss investigation


In [ ]:

#Top 10 Loss-Making Products using analysis_df
product_analysis = analysis_df.groupby(
    ["Product ID", "Product Name"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum"),
    Total_Orders=("Order ID", "nunique")
)

product_analysis["Profit_Margin_%"] = (
    product_analysis["Total_Profit"] /
    product_analysis["Total_Sales"] * 100
)

loss_products = product_analysis[
    product_analysis["Total_Profit"] < 0
].sort_values("Total_Profit")

loss_products.head(10).round(2)



## Discount and profitability


In [ ]:
#Discount Analysis
discount_analysis = analysis_df.groupby("Discount").agg(
    Total_Sales=("Sales", "sum"),   
    Total_Profit=("Profit", "sum"),
    Average_Profit=("Profit", "mean"),  
    Total_Orders=("Order ID", "nunique"),
    Loss_Orders=("Profit", lambda x: (x < 0).sum())
)

discount_analysis["Profit_Margin_%"] = (
    discount_analysis["Total_Profit"] /
    discount_analysis["Total_Sales"]* 100 

)
discount_analysis.round(2)


### Findings from the selected reporting period
The summary below is calculated from the tables above, so it stays aligned with the selected cutoff and source data.

In [ ]:
top_category = category_analysis["Total_Profit"].idxmax()
lowest_subcategory = subcategory_analysis["Total_Profit"].idxmin()
negative_discount_levels = discount_analysis.index[discount_analysis["Total_Profit"] < 0].tolist()

print(f"Category with highest total profit: {top_category} (${category_analysis.loc[top_category, 'Total_Profit']:,.2f})")
print(f"Sub-category with lowest total profit: {lowest_subcategory} (${subcategory_analysis.loc[lowest_subcategory, 'Total_Profit']:,.2f})")
if negative_discount_levels:
    print("Discount levels with negative aggregate profit:", ", ".join(f"{level:.0%}" for level in negative_discount_levels))
else:
    print("No discount level has negative aggregate profit in the selected period.")
print("Interpret discount-level patterns alongside transaction counts and product mix; this comparison is descriptive, not causal.")


### Discount patterns among loss-making products


In [ ]:
#Discount analysis by loss-making product
loss_product_discount = analysis_df[
    analysis_df["Product ID"].isin(
        loss_products.head(10).index.get_level_values("Product ID")
    )
].groupby(["Product ID","Product Name", "Discount"]).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum")
)

loss_product_discount.round(2)


### High-discount sub-category review


In [ ]:
discount_analysis = analysis_df.groupby("Discount").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Records=("Discount", "size"),
    Orders=("Order ID", "nunique")
)

discount_analysis["Profit_Margin_%"] = (
    discount_analysis["Total_Profit"]
    / discount_analysis["Total_Sales"] * 100
)

display(discount_analysis.round(2))


In [ ]:
high_discount_analysis = (
    analysis_df[analysis_df["Discount"] >= 0.30]
    .groupby(["Discount", "Sub-Category"])
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Records=("Discount", "size")
    )
)

display(high_discount_analysis.sort_values("Total_Profit").round(2))


## Customer segment performance


In [ ]:
#Customer Segment Analysis
segment_analysis = analysis_df.groupby("Segment").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique")
)

segment_analysis["Profit_Margin_%"] = (
    segment_analysis["Total_Profit"] /
    segment_analysis["Total_Sales"] * 100
)

segment_analysis.sort_values(
    "Total_Profit", ascending=False
).round(2)


## Regional performance


In [ ]:
#Regional Analysis
region_analysis = analysis_df.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique")
)

region_analysis["Profit_Margin_%"] = (
    region_analysis["Total_Profit"] /
    region_analysis["Total_Sales"] * 100
)

region_analysis.sort_values(
    "Total_Profit", ascending=False
).round(2)


## Monthly performance


In [ ]:
#Monthly Sales Trends
analysis_df["Order Date"] = pd.to_datetime(analysis_df["Order Date"])
monthly_analysis = analysis_df.groupby(
    analysis_df["Order Date"].dt.to_period("M")
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique")
)

monthly_analysis.tail(12).round(2)


### Monthly sales trend


In [ ]:
monthly_analysis["Total_Sales"].plot(
    kind="line",
    figsize=(12, 5),
    marker="o"
)

plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


### Monthly profit trend


In [ ]:
monthly_analysis["Total_Profit"].plot(
    kind="line",
    figsize=(12, 5),
    marker="o"
)

plt.title("Monthly Profit Trend")
plt.xlabel("Month")
plt.ylabel("Total Profit ($)")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


### Shipping measure note
The source records ship date, not delivery date. The interval shown is order-to-ship time, and shipping costs are unavailable, so this section cannot assess end-to-end delivery or shipping cost efficiency.

## Shipping mode comparison


In [ ]:
#Shipping analysis
analysis_df["Ship Date"] = pd.to_datetime(analysis_df["Ship Date"])
analysis_df["Order Date"] = pd.to_datetime(analysis_df["Order Date"])

analysis_df["Delivery_Days"] = (
    analysis_df["Ship Date"] - analysis_df["Order Date"]
).dt.days
shipping_analysis = analysis_df.groupby("Ship Mode").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique"),
    Average_Delivery_Days=("Delivery_Days", "mean")
)

shipping_analysis["Profit_Margin_%"] = (
    shipping_analysis["Total_Profit"] /
    shipping_analysis["Total_Sales"] * 100
)

shipping_analysis.round(2)


## Customer profitability


In [ ]:
##Calculate customer performance
customer_analysis = analysis_df.groupby(
    ["Customer ID", "Customer Name"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique"),
    Total_Quantity=("Quantity", "sum")
)

customer_analysis["Profit_Margin_%"] = (
    customer_analysis["Total_Profit"] /
    customer_analysis["Total_Sales"] * 100
)

top_customers = customer_analysis.sort_values(
    "Total_Sales",
    ascending=False
).head(10)

top_customers.round(2)


In [ ]:
top_profit_customers = customer_analysis.sort_values(
    "Total_Profit",
    ascending=False
).head(10)

top_profit_customers.round(2)


In [ ]:
bottom_profit_customers = customer_analysis.sort_values(
    "Total_Profit",
    ascending=True
).head(10)

bottom_profit_customers.round(2)


## Final KPI check and reconciliation


In [ ]:
##Final project data validation
total_sales = analysis_df["Sales"].sum()
total_profit = analysis_df["Profit"].sum()
total_orders = analysis_df["Order ID"].nunique()

profit_margin = total_profit / total_sales * 100

print("Total Records:", len(analysis_df))
print("Unique Orders:", total_orders)
print("Total Sales: $", round(total_sales, 2))
print("Total Profit: $", round(total_profit, 2))
print("Profit Margin:", round(profit_margin, 2), "%")


## Portfolio charts


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Profit by category
category_profit = category_analysis["Total_Profit"]
category_colors = [
    "seagreen" if value >= 0 else "indianred"
    for value in category_profit
]

category_profit.plot(
    kind="bar",
    ax=axes[0],
    color=category_colors,
    title="Total Profit by Category"
)
axes[0].set_ylabel("Profit ($)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)
axes[0].axhline(0, color="black", linewidth=0.8)

# Profit by discount level
discount_profit = discount_analysis["Total_Profit"]
discount_colors = [
    "seagreen" if value >= 0 else "indianred"
    for value in discount_profit
]

discount_profit.plot(
    kind="bar",
    ax=axes[1],
    color=discount_colors,
    title="Total Profit by Discount Level"
)
axes[1].set_ylabel("Profit ($)")
axes[1].set_xlabel("Discount")
axes[1].tick_params(axis="x", rotation=45)
axes[1].axhline(0, color="black", linewidth=0.8)

plt.tight_layout()
chart_dir = Path("notebooks") if Path("notebooks").is_dir() else Path(".")
plt.savefig(chart_dir / "portfolio_charts.png", dpi=300, bbox_inches="tight")
plt.show()

